# Synthetic Data Generator
### Step 11a : Build Synthetic Final Aligned Output Stage

This notebook builds the final Kafka/Postgres synthetic output so it matches the Kaggle-style dataset shape:

- `dataset_id`
- `run_id`
- `asset_id`
- `timestamp`
- `sensor_00` ... `sensor_51`
- `machine_status`

It uses the rebuilt stage as the source table and does **not** read premelt for this final output step.


In [ ]:
# -----------------------------------------------------------------------------
# Stage 11B — Compact Synthetic Final Output
# -----------------------------------------------------------------------------
# Purpose:
# Build a Kaggle-like compact final output table.
#
# Source:
# - synthetic_sensor_observations_rebuilt_stage
#
# Target:
# - synthetic_sensor_observations_final_output

In [ ]:
import os
import pandas as pd 

from utils.database.postgres import (
    get_engine_from_env, 
    read_sql_dataframe,
    execute_sql,
)

import pandas as pd

from utils.database.postgres import execute_sql, read_sql_dataframe

from utils.synthetic.pipeline.final_aligned_observation_writer import (
    build_final_aligned_observations_stage,
)

from utils.synthetic.pipeline.final_aligned_output_writer import build_synthetic_final_aligned_output_stage

from utils.synthetic.pipeline.final_aligned_incremental import run_final_align_loop

from utils.core.env_helpers import (
    env_bool, 
    env_int, 
    env_optional_int, 
    env_str,
)

pd.set_option("display.max_colwidth", None)



In [ ]:
SCHEMA = env_str("CAPSTONE_SCHEMA", "capstone")

DATASET_ID = env_str(
    "SYNTHETIC_DATASET_ID",
    "pump_synthetic_v1",
    aliases=("DATASET_ID",),
)

RUN_ID = env_str(
    "SYNTHETIC_RUN_ID",
    "synthetic_run_001",
    aliases=("RUN_ID",),
)

NUMBER_OF_SENSORS = env_int("SYNTHETIC_SENSOR_COUNT", 52)

IF_EXISTS_FLAG = "replace"

COMPLETE_ONLY_FLAG = env_bool(
    "SYNTHETIC_FINAL_ALIGN_COMPLETE_ONLY",
    True,
)

OBSERVATION_WINDOW_SIZE = env_int(
    "SYNTHETIC_REBUILD_OBSERVATION_WINDOW_SIZE",
    2500,
)

REBUILT_TABLE_NAME = env_str(
    "SYNTHETIC_REBUILT_OBSERVATIONS_TABLE",
    "synthetic_sensor_observations_rebuilt_stage",
)

FINAL_OUTPUT_TABLE_NAME = env_str(
    "SYNTHETIC_FINAL_OUTPUT_TABLE",
    "synthetic_sensor_observations_final_output",
)

N_SENSORS = NUMBER_OF_SENSORS

TIMESTAMP_OUTPUT_COLUMN = "timestamp"

MACHINE_STATUS_OUTPUT_COLUMN = "machine_status"

STATUS_MAPPING = {
    "normal": "NORMAL",
    "broken": "BROKEN",
    "abnormal": "BROKEN",
    "failure": "BROKEN",
    "failed": "BROKEN",
    "fault": "BROKEN",
    "recovering": "RECOVERING",
    "recovery": "RECOVERING",
}

print("Final aligned config")
print("schema:", SCHEMA)
print("dataset id:", DATASET_ID)
print("run id:", RUN_ID)
print("rebuilt table:", REBUILT_TABLE_NAME)
print("final output table:", FINAL_OUTPUT_TABLE_NAME)
print("batch size:", OBSERVATION_WINDOW_SIZE)

Final aligned config
schema: capstone
dataset id: pump_synthetic_v1
run id: synthetic_run_001
premelt table: synthetic_observations_premelt_stage
rebuilt table: synthetic_sensor_observations_rebuilt_stage
target table: synthetic_sensor_observations_final_aligned_stage
batch size: 1000
max iterations: None


---

In [9]:
engine = get_engine_from_env()

---

In [ ]:
stage_11b_final_output_columns_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        ordinal_position,
        column_name,
        data_type
    FROM information_schema.columns
    WHERE table_schema = :schema_name
      AND table_name = :table_name
    ORDER BY ordinal_position
    """,
    params={
        "schema_name": SCHEMA,
        "table_name": FINAL_OUTPUT_TABLE_NAME,
    },
)

display(stage_11b_final_output_columns_dataframe)

,ordinal_position,column_name,data_type
0,1,aligned_observation_id,bigint
1,2,dataset_id,text
2,3,run_id,text
3,4,asset_id,text
4,5,event_time,timestamp with time zone
5,6,event_step,bigint
6,7,time_index,bigint
7,8,machine_status,text
8,9,operational_state,text
9,10,anomaly_flag,integer


In [ ]:
'''
execute_sql(
    engine,
    f"""
    DROP TABLE IF EXISTS "{SCHEMA}"."{TARGET_TABLE_NAME}" CASCADE
    """
)

print(f"Dropped stale Stage 11 target table: {SCHEMA}.{TARGET_TABLE_NAME}")
'''

Dropped stale Stage 11 target table: capstone.synthetic_sensor_observations_final_aligned_stage


In [ ]:
# -----------------------------------------------------------------------------
# Build compact Kaggle-like synthetic output
# -----------------------------------------------------------------------------

stage_11b_result = build_synthetic_final_aligned_output_stage(
    engine=engine,
    schema=SCHEMA,
    rebuilt_table=REBUILT_TABLE_NAME,
    target_table=FINAL_OUTPUT_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    n_sensors=N_SENSORS,
    complete_only=True,
    if_exists="replace",
    observation_window_size=OBSERVATION_WINDOW_SIZE,
    timestamp_output_column=TIMESTAMP_OUTPUT_COLUMN,
    machine_status_output_column=MACHINE_STATUS_OUTPUT_COLUMN,
)

display(pd.DataFrame([stage_11b_result]))

[final-align-incremental] Added 54 new columns to capstone.synthetic_sensor_observations_final_aligned_stage


,status,final_align_token,claimed_count,written_count,skipped_existing_count,target_table
0,completed,e009d04d-ca6c-410c-9fe8-dc0d44d66321,1000,1000,0,synthetic_sensor_observations_final_aligned_stage
1,completed,6f0e4c4e-8d5a-4505-8d27-f84e7c5b9b52,1000,1000,0,synthetic_sensor_observations_final_aligned_stage
2,completed,bac4eb36-8929-4b9b-9ae6-878a71f8dcfa,1000,1000,0,synthetic_sensor_observations_final_aligned_stage
3,completed,6e1a3dd7-c872-4902-80dd-b609c8a9484f,1000,1000,0,synthetic_sensor_observations_final_aligned_stage
4,completed,850a90ce-9cd7-4c9f-8539-d782caef0ab2,1000,1000,0,synthetic_sensor_observations_final_aligned_stage
...,...,...,...,...,...,...
220,completed,5c73f364-6004-4b15-8cbb-40194fec6b1e,1000,1000,0,synthetic_sensor_observations_final_aligned_stage
221,completed,253a47ec-7382-4a65-9c7b-af896dceafec,1000,1000,0,synthetic_sensor_observations_final_aligned_stage
222,completed,95556252-2023-4f05-be2a-7d4d5cf13a7c,1000,1000,0,synthetic_sensor_observations_final_aligned_stage
223,completed,7b05ba70-23a5-4a1b-8e60-32f4937f5e01,1000,1000,0,synthetic_sensor_observations_final_aligned_stage


----

In [ ]:
# -----------------------------------------------------------------------------
# Validate compact final output
# -----------------------------------------------------------------------------

stage_11b_validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT dataset_id) AS dataset_id_count,
        COUNT(DISTINCT run_id) AS run_id_count,
        COUNT(DISTINCT asset_id) AS asset_id_count,
        COUNT(*) FILTER (WHERE "{TIMESTAMP_OUTPUT_COLUMN}" IS NULL) AS null_timestamp_count,
        COUNT(*) FILTER (WHERE "{MACHINE_STATUS_OUTPUT_COLUMN}" IS NULL) AS null_machine_status_count,
        MIN("{TIMESTAMP_OUTPUT_COLUMN}") AS min_timestamp,
        MAX("{TIMESTAMP_OUTPUT_COLUMN}") AS max_timestamp,
        COUNT(*) FILTER (WHERE "{MACHINE_STATUS_OUTPUT_COLUMN}" = 'NORMAL') AS normal_rows,
        COUNT(*) FILTER (WHERE "{MACHINE_STATUS_OUTPUT_COLUMN}" = 'BROKEN') AS broken_rows,
        COUNT(*) FILTER (WHERE "{MACHINE_STATUS_OUTPUT_COLUMN}" = 'RECOVERING') AS recovering_rows,
        (
            COUNT(*) = 225000
            AND COUNT(*) FILTER (WHERE "{TIMESTAMP_OUTPUT_COLUMN}" IS NULL) = 0
            AND COUNT(*) FILTER (WHERE "{MACHINE_STATUS_OUTPUT_COLUMN}" IS NULL) = 0
        ) AS ready_for_export
    FROM "{SCHEMA}"."{FINAL_OUTPUT_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(stage_11b_validation_dataframe)

---


### Sample Output

This shows the final Kaggle-shaped output table layout.


In [ ]:
stage_11b_sample_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        dataset_id,
        run_id,
        asset_id,
        "{TIMESTAMP_OUTPUT_COLUMN}",
        sensor_00,
        sensor_01,
        sensor_02,
        sensor_03,
        sensor_04,
        "{MACHINE_STATUS_OUTPUT_COLUMN}"
    FROM "{SCHEMA}"."{FINAL_OUTPUT_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    ORDER BY "{TIMESTAMP_OUTPUT_COLUMN}", dataset_id, run_id, asset_id
    LIMIT 10
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(stage_11b_sample_dataframe)

---

In [ ]:
stage_11b_status_distribution_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        "{MACHINE_STATUS_OUTPUT_COLUMN}" AS machine_status,
        COUNT(*) AS row_count
    FROM "{SCHEMA}"."{FINAL_OUTPUT_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    GROUP BY "{MACHINE_STATUS_OUTPUT_COLUMN}"
    ORDER BY "{MACHINE_STATUS_OUTPUT_COLUMN}"
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(stage_11b_status_distribution_dataframe)

---

In [ ]:
# -----------------------------------------------------------------------------
# Preview compact final output
# -----------------------------------------------------------------------------

preview_final_output_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT *
    FROM "{SCHEMA}"."{FINAL_OUTPUT_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    ORDER BY "{TIMESTAMP_OUTPUT_COLUMN}"
    LIMIT 10
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(preview_final_output_dataframe)

---

In [ ]:
# Dispose Engine
engine.dispose()